In [1]:
import os
import sys
import glob
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from omegaconf import OmegaConf

import wandb
import pandas as pd
import re

from torchtune import config

from training import SelfPredictionTrainingRecipeDistributed
from evaluation.pfa_evaluation import (process_data, 
                                        compute_losses_per_level, 
                                        compute_losses_per_level_statistics)

api = wandb.Api()
cfg = OmegaConf.load(f'{PROJECT_ROOT}/configs/llama_0.1B_PHi.yaml')

In [18]:
runs = api.runs("hidden-state-predictions/llama-0.1B-gumbel-layers")
print(f'Runs: {len(runs)}')


runs_list = []
for run in runs:
    # Logic to split the name: seed-42-layer-3-llf-1e-4
    # Splitting by '-' and picking indices
    parts = run.name.split('-')
    
    runs_list.append({
        "seed": parts[1],
        "layer": parts[3],
        "ID": run.id
    })

df = pd.DataFrame(runs_list)

Runs: 24


# initialize dataset and model

In [3]:
included_levels = [0, 1, 2, 3, 4]
num_datapoints = 10
batch_size = 8
base_dir = '/home/woody/iwbi/iwbi106h/suuraj/models/self-prediction-models'

cfg_tokenizer = cfg.tokenizer
tokenizer = config.instantiate(cfg.tokenizer)

cfg_dataset = cfg.dataset
cfg_dataset.included_learning_levels = included_levels
packed_on_the_fly = cfg_dataset.pop("packed_on_the_fly", False)
packed_sequence_length = cfg_dataset.pop("packed_sequence_length", 2048)
split_across_pack = cfg_dataset.pop("split_across_pack", False)
num_workers = cfg_dataset.pop("num_workers", 8)

ds = config.instantiate(cfg_dataset, tokenizer)

In [4]:
results_dict = []
losses = ['next_token_losses', 'latent_losses0', 'latent_entropy0', 'phi_losses0']

In [6]:
for id_data in zip(df['seed'], df['layer'], df['ID']):
    base_model_path = glob.glob(os.path.join(base_dir, f'*{id_data[2]}'))[0]
    assert os.path.exists(base_model_path)
    cfg = OmegaConf.load(os.path.join(base_model_path, 'config.yaml'))
    cfg.checkpointer.checkpoint_dir = base_model_path
    cfg.checkpointer.checkpoint_files = ["torchtune_model_last.pt"]
    cfg.train_from_scratch = False
    cfg.metric_logger.mode = 'disabled'
    
    recipe = SelfPredictionTrainingRecipeDistributed(cfg=cfg)
    recipe.setup(cfg=cfg)

    recipe._model.eval()
    print(f'Evaluating {id_data[2]}...')
    datapoints = process_data(recipe, num_datapoints, ds, batch_size)
    losses_vs_learning_levels = compute_losses_per_level(datapoints,filter_out_spaces=False,losses=losses,levels=included_levels)
    losses_vs_learning_levels_statistics = compute_losses_per_level_statistics(losses_vs_learning_levels,losses=losses,levels=included_levels)
    results_dict.append({'layer': id_data[1],
                         'seed': id_data[0],
                         'statistics': losses_vs_learning_levels_statistics,})
results_dict_df = pd.DataFrame(results_dict)

In [10]:
# next_token_losses', 'latent_losses', 'latent_entropy', 'phi_losses']
aggregated_results = []

for id_data in zip(results_dict_df['seed'], results_dict_df['layer'], results_dict_df['statistics']):
    for loss_name, levels_dict in losses_vs_learning_levels_statistics.items():
        for level, stats_data in levels_dict.items():
            aggregated_results.append({
                'layer': id_data[1],
                'seed': id_data[0],
                'loss': loss_name,
                'level': level,
                'mean': stats_data['mean']
            })

In [20]:
aggregated_results_df = pd.DataFrame(aggregated_results)
with pd.option_context('display.max_rows', None, 'display.max_columns', None): 
    print(aggregated_results_df)

    layer seed               loss  level      mean
0       3   42  next_token_losses      0  0.789630
1       3   42  next_token_losses      1  1.357209
2       3   42  next_token_losses      2  1.733989
3       3   42  next_token_losses      3  2.057108
4       3   42  next_token_losses      4  2.974084
5       3   42     latent_losses0      0  5.366234
6       3   42     latent_losses0      1  5.366623
7       3   42     latent_losses0      2  4.445365
8       3   42     latent_losses0      3  3.991647
9       3   42     latent_losses0      4  0.695135
10      3   42    latent_entropy0      0  2.950374
11      3   42    latent_entropy0      1  2.949892
12      3   42    latent_entropy0      2  3.871473
13      3   42    latent_entropy0      3  4.324343
14      3   42    latent_entropy0      4  7.618549
15      3   42        phi_losses0      0  1.163502
16      3   42        phi_losses0      1  1.597525
17      3   42        phi_losses0      2  1.958765
18      3   42        phi_losse

In [17]:
test_group = aggregated_results_df[
    (aggregated_results_df['layer'] == '3') & (aggregated_results_df['level'] == 0)
]
print(test_group[['seed', 'mean']])

    seed      mean
0     42  0.789630
5     42  5.366234
10    42  2.950374
15    42  1.163502
60    44  0.789630
65    44  5.366234
70    44  2.950374
75    44  1.163502
220   40  0.789630
225   40  5.366234
230   40  2.950374
235   40  1.163502


In [15]:
summary_std = aggregated_results_df.groupby(['layer', 'loss', 'level'])['mean'].agg(
    avg_across_seeds='mean',
    std_across_seeds='std'
).reset_index()


summary_std['report_string'] = summary_std.apply(
    lambda x: f"{x['avg_across_seeds']:.4f} ± {x['std_across_seeds']:.4f}", axis=1
)

print(summary_std)

    layer             loss  level  avg_across_seeds  std_across_seeds  \
0       1  latent_entropy0      0          2.950374               0.0   
1       1  latent_entropy0      1          2.949892               0.0   
2       1  latent_entropy0      2          3.871473               0.0   
3       1  latent_entropy0      3          4.324343               0.0   
4       1  latent_entropy0      4          7.618549               0.0   
..    ...              ...    ...               ...               ...   
135     9      phi_losses0      0          1.163502               NaN   
136     9      phi_losses0      1          1.597525               NaN   
137     9      phi_losses0      2          1.958765               NaN   
138     9      phi_losses0      3          1.866534               NaN   
139     9      phi_losses0      4          0.553934               NaN   

       report_string  
0    2.9504 ± 0.0000  
1    2.9499 ± 0.0000  
2    3.8715 ± 0.0000  
3    4.3243 ± 0.0000  
4    7.6

In [16]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(summary_std)


    layer               loss  level  avg_across_seeds  std_across_seeds  \
0       1    latent_entropy0      0          2.950374               0.0   
1       1    latent_entropy0      1          2.949892               0.0   
2       1    latent_entropy0      2          3.871473               0.0   
3       1    latent_entropy0      3          4.324343               0.0   
4       1    latent_entropy0      4          7.618549               0.0   
5       1     latent_losses0      0          5.366234               0.0   
6       1     latent_losses0      1          5.366623               0.0   
7       1     latent_losses0      2          4.445365               0.0   
8       1     latent_losses0      3          3.991647               0.0   
9       1     latent_losses0      4          0.695134               0.0   
10      1  next_token_losses      0          0.789630               0.0   
11      1  next_token_losses      1          1.357209               0.0   
12      1  next_token_los